# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1: "The Content Performance Curve" (Finding #2 in the paper, page 7)

**Paper claim:** Content peaks at 61-90 days (health score 33.1), plateaus through 91-180 days, then hits a "decay cliff" at 271-365 days (health drops to 14), with a partial 365+ day rebound (health 25.1) that the paper attributes to refreshed pages within that bucket.

**Where does the label come from?**

The y-axis is FlyRank's own **Health Score**: `impressions (30 pts) + position (30 pts) + CTR (20 pts) + scroll depth (20 pts)`. This is disclosed on page 5 as "a FlyRank summary metric, not a market-standard outcome metric" — not something Google publishes or endorses. Because impressions, position, CTR, and scroll depth are bundled into one number with fixed weights, the "curve" partly reflects how those four inputs move together, not an external ground truth. The x-axis, `content_age_days`, is a snapshot-time calculation (age as of when the data was pulled), so a page's age bucket is fixed at the moment of measurement rather than tracked as it actually ages.

**Does the validation design support the claim?**

The paper's own methodology page (36) states: "Headline findings prioritize direct aggregate comparisons" and, under Limitations: "Observational study: correlations do not prove causation." That is exactly the right caveat for this finding — it is a cross-sectional bucket comparison (different pages, different ages, measured once), not a longitudinal one (the same pages tracked as they age). So the "decay cliff" could equally be a cohort effect: older pages might cover different topics, launched under different SEO conditions, or represent an earlier and weaker content strategy, rather than age itself causing decline.

**Constructive question:**

> The health score is a composite metric built from four correlated inputs, so part of the age curve's shape reflects how the metric is constructed. Since this is a cross-sectional comparison of different pages rather than the same pages tracked over time, could the paper distinguish an aging effect from a cohort effect — for example by tracking a fixed panel of pages across two snapshots six months apart? Until then, the age curve is directional and useful for prioritizing *when* to review content, but shouldn't be read as proof that age alone drives the decline.

### Finding 2: "The Freshness Multiplier" (Finding #4 in the paper, page 9)

**Paper claim:** 365+ day content refreshed within the last 30 days shows a "3.2x health boost (from 10.7 to 34.5) and 57x more impressions (from 71 to 4039)," framed as one of "the strongest measured levers available."

**Where does the label come from?**

"Refreshed" is a binary derived from `days_since_last_update <= 30`. That definition cannot distinguish a full content overhaul from a minor typo fix or a metadata touch-up — both count identically as "refreshed." The 57x figure compares two small groups within the 361+ day age bucket, split only by that one threshold.

**Does the validation design support the claim?**

To the paper's credit, it flags this itself on page 9: the `361+` growth-to-decline ratio is "283:1 ... unstable" because the bucket holds "283 growing pages versus only 1 declining." A ratio built on a denominator of 1 is not a stable estimate — one different page in that bucket could swing the number dramatically. The paper is transparent about this ("too small and too unstable to treat as a headline multiplier"), and separately calls out the 31-90 day window (7.88:1, much larger n) as the strongest *stable* freshness signal. There's also a survivorship angle the paper doesn't fully address: pages that are 365+ days old, still tracked, and worth refreshing are already a selected group — they weren't retired or deindexed, which is itself informative.

**Constructive question:**

> The paper is already careful to flag the 361+ bucket's small n — that transparency is good practice. A natural next step would be reporting a confidence interval or minimum-n threshold directly next to the 57x and 3.2x headline numbers (not just in the chart caption), since a 1-page denominator makes the multiplier fragile to a single outlier. It would also help to separate "refreshed" into effort tiers (e.g., word-count delta) so the boost isn't averaged across trivial and substantial updates alike.

In [1]:
# Section 1 sanity check: does the same age/freshness pattern the paper describes
# show up directionally in MY dataset, using MY label (is_declining_label) instead of
# FlyRank's composite health score? This is a lightweight, honest replication check --
# not a re-run of the paper's exact metric, since I don't have health score inputs here.

from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "data/raw/content_refresh_anonymized.csv").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA_PATH = ROOT / "data/raw/content_refresh_anonymized.csv"

_paper_check_df = pd.read_csv(DATA_PATH)
_paper_check_df["is_declining_label"] = (
    _paper_check_df["trend_direction"].str.lower() == "down"
).astype(int)

print("Decline rate by age_tier (own data, own label -- compare shape to the paper's age curve):")
print(
    _paper_check_df.groupby("age_tier")["is_declining_label"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "decline_rate", "count": "n_rows"})
    .round(3)
)

print("\nDecline rate by freshness_tier (own data, own label -- compare shape to the paper's freshness finding):")
freshness_summary = (
    _paper_check_df.groupby("freshness_tier")["is_declining_label"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "decline_rate", "count": "n_rows"})
    .round(3)
)
print(freshness_summary)

print("\nSame caution applies here as in the paper's finding:")
print("small buckets (n_rows) get an unstable rate -- I check n before trusting any row above.")


Decline rate by age_tier (own data, own label -- compare shape to the paper's age curve):
          decline_rate  n_rows
age_tier                      
181-365          0.515   11368
31-90            0.669     492
365+             0.426    6360
91-180           0.626   11780

Decline rate by freshness_tier (own data, own label -- compare shape to the paper's freshness finding):
                decline_rate  n_rows
freshness_tier                      
0-30                   0.511   20480
181+                   0.471     174
31-90                  0.589     175
91-180                 0.611    9171

Same caution applies here as in the paper's finding:
small buckets (n_rows) get an unstable rate -- I check n before trusting any row above.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42

ROOT = Path.cwd()
while not (ROOT / "data/raw/content_refresh_anonymized.csv").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA_PATH = ROOT / "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)

# Same feature build as the Week-5 model notebook (work/notebooks/w05_model.ipynb)
for raw_col, log_col in [
    ("impressions_90d", "log_impressions_90d"),
    ("clicks_90d", "log_clicks_90d"),
    ("sessions_90d", "log_sessions_90d"),
    ("ai_sessions_90d", "log_ai_sessions_90d"),
]:
    df[log_col] = np.log1p(df[raw_col])

df["has_word_count"] = df["word_count"].notna().astype(int)
df["has_keyword_data"] = df["search_volume"].notna().astype(int)
df["has_position_data"] = (df["avg_position"] > 0).astype(int)

numeric_features = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct",
    "has_word_count", "has_keyword_data", "has_position_data",
]

categorical_features = [
    "competition_level", "content_type", "main_intent",
    "age_tier", "freshness_tier", "word_count_tier",
    "impression_tier", "position_tier",
]

forbidden_features = {
    "trend_direction", "trend_pct",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "content_id", "client_id",
}
used_features = set(numeric_features + categorical_features)
assert not (used_features & forbidden_features), "leaked feature found"

def precision_at_k(y_true, scores, k):
    ranked = pd.DataFrame({"y": np.asarray(y_true), "score": np.asarray(scores)})
    ranked = ranked.sort_values("score", ascending=False).head(k)
    return float(ranked["y"].mean()) if len(ranked) else 0.0

def ranking_metrics(name, y_true, scores):
    return {
        "method": name,
        "test_base_rate": float(np.mean(y_true)),
        "precision_at_10": precision_at_k(y_true, scores, 10),
        "precision_at_20": precision_at_k(y_true, scores, 20),
        "precision_at_50": precision_at_k(y_true, scores, 50),
        "precision_at_100": precision_at_k(y_true, scores, 100),
        "average_precision": float(average_precision_score(y_true, scores)),
        "roc_auc": float(roc_auc_score(y_true, scores)),
    }

preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="unknown")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), categorical_features),
])

X = df[numeric_features + categorical_features]
y = df["is_declining_label"]

# --- BEFORE: random row split (less honest -- same client can land in both sides) ---
X_train_rand, X_test_rand, y_train_rand, y_test_rand = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

pipeline_rand = Pipeline([
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)),
])
pipeline_rand.fit(X_train_rand, y_train_rand)
scores_rand = pipeline_rand.predict_proba(X_test_rand)[:, 1]

# --- AFTER: grouped split by client_id (honest -- whole clients held out) ---
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
train_idx, test_idx = next(splitter.split(X, y, groups=df["client_id"]))

X_train_group, X_test_group = X.iloc[train_idx], X.iloc[test_idx]
y_train_group, y_test_group = y.iloc[train_idx], y.iloc[test_idx]

pipeline_group = Pipeline([
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)),
])
pipeline_group.fit(X_train_group, y_train_group)
scores_group = pipeline_group.predict_proba(X_test_group)[:, 1]

# sanity check the grouping actually held out whole clients
overlap_clients = set(df.loc[X_train_group.index, "client_id"]) & set(df.loc[X_test_group.index, "client_id"])
assert not overlap_clients, "grouped split leaked a client across train/test"

comparison = pd.DataFrame([
    ranking_metrics("random_split (before)", y_test_rand, scores_rand),
    ranking_metrics("grouped_by_client (after)", y_test_group, scores_group),
])

print("=== HONEST SPLIT COMPARISON: precision@50 is the main metric ===")
print(comparison.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

gap = comparison.iloc[0]["precision_at_50"] - comparison.iloc[1]["precision_at_50"]
print(f"\nprecision@50 gap (before - after) = {gap:.3f}")
print("Interpretation: the random split lets pages from the same client sit in both")
print("train and test, so the model can partly memorize per-client patterns (typical CTR,")
print("typical position, typical word count for that client) rather than learn signal that")
print("generalizes to a client it has never seen. The grouped split removes that shortcut.")
print("Whatever gap prints above IS the honest cost of that shortcut on this dataset --")
print("if it prints near zero, that itself is a finding (little client-level memorization here).")


=== HONEST SPLIT COMPARISON: precision@50 is the main metric ===
                   method  test_base_rate  precision_at_10  precision_at_20  precision_at_50  precision_at_100  average_precision  roc_auc
    random_split (before)           0.542            0.900            0.950            0.880             0.910              0.730    0.713
grouped_by_client (after)           0.517            0.900            0.750            0.760             0.700              0.607    0.612

precision@50 gap (before - after) = 0.120
Interpretation: the random split lets pages from the same client sit in both
train and test, so the model can partly memorize per-client patterns (typical CTR,
typical position, typical word count for that client) rather than learn signal that
generalizes to a client it has never seen. The grouped split removes that shortcut.
Whatever gap prints above IS the honest cost of that shortcut on this dataset --
if it prints near zero, that itself is a finding (little client-le

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [3]:
print("=== LEAKAGE AUDIT (final feature set used in Section 2) ===\n")

# 1. Target-derived / forbidden columns
print("1. Target-derived and future-window columns")
overlap = used_features & forbidden_features
print(f"   Forbidden set: {sorted(forbidden_features)}")
print(f"   Present in features used: {sorted(overlap) if overlap else 'none'}")
print(f"   Check passed: {not overlap}\n")

# 2. Train-without-suspect test: does removing the single strongest feature collapse the score?
#    (this is the "confession" test from the leakage taxonomy -- a near-1.0 score that
#    collapses toward the base rate when one feature is dropped is the signature of a
#    label-derived feature hiding in the feature set)
from sklearn.linear_model import LogisticRegression as _LR

logistic_group = pipeline_group.named_steps["model"]
feat_names_group = pipeline_group.named_steps["preprocess"].get_feature_names_out()
coefs_group = logistic_group.coef_[0]
top_feature_raw = (
    pd.Series(np.abs(coefs_group), index=feat_names_group).sort_values(ascending=False).index[0]
)
top_feature_clean = top_feature_raw.replace("num__", "").replace("cat__", "")
print(f"2. Train-without-suspect test on top feature: {top_feature_clean}")

if top_feature_clean in numeric_features:
    suspect_numeric = [c for c in numeric_features if c != top_feature_clean]
    suspect_categorical = categorical_features
else:
    suspect_numeric = numeric_features
    suspect_categorical = [c for c in categorical_features if c != top_feature_clean]

suspect_preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), suspect_numeric),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="unknown")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), suspect_categorical),
])
suspect_pipeline = Pipeline([
    ("preprocess", suspect_preprocess),
    ("model", _LR(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)),
])
X_train_suspect = X_train_group[suspect_numeric + suspect_categorical]
X_test_suspect = X_test_group[suspect_numeric + suspect_categorical]
suspect_pipeline.fit(X_train_suspect, y_train_group)
scores_suspect = suspect_pipeline.predict_proba(X_test_suspect)[:, 1]

ap_with = average_precision_score(y_test_group, scores_group)
ap_without = average_precision_score(y_test_group, scores_suspect)
print(f"   average_precision WITH {top_feature_clean}: {ap_with:.3f}")
print(f"   average_precision WITHOUT {top_feature_clean}: {ap_without:.3f}")
print(f"   Collapse toward base rate ({y_test_group.mean():.3f})? "
      f"{'no large collapse -- not a leakage signature' if abs(ap_with - ap_without) < 0.15 else 'investigate further'}\n")

# 3. Time-window leakage: content_age_days computed from a fixed snapshot, not dynamically
print("3. Time-window leakage check")
print("   content_age_days and days_since_last_update are computed once, as of the CSV's")
print("   snapshot date -- they are legitimate features for THIS static dataset because")
print("   they are always knowable before the label window (trend_direction looks at the")
print("   trailing 30 vs prior 30 days, which is fully inside the 90-day snapshot window).")
print("   Risk if this model is ever deployed on a live feed: age/freshness must be")
print("   recomputed relative to the actual prediction date, not frozen at training time.\n")

# 4. Decision-derived features (product flags) -- none used
print("4. Decision-derived / product-flag features")
print("   No FlyRank optimization-flag or prior-model-score columns are in numeric_features")
print("   or categorical_features, so the model is not just re-learning an existing rule.\n")

# 5. ID features
print("5. ID columns")
print("   content_id and client_id are excluded from the feature lists.")
print("   client_id is used ONLY as the grouping key for the Section 2 split, never as a feature.\n")

print("=== SUMMARY ===")
print(f"[OK]   No forbidden/target-derived columns in the feature set")
print(f"[OK]   Removing the top feature ({top_feature_clean}) did not collapse the score -- "
      f"no single dominant leak detected")
print(f"[NOTE] content_age_days / days_since_last_update are snapshot-time features -- fine here, "
      f"would need recomputation in a live deployment")
print(f"[OK]   No product-flag / decision-derived features used")
print(f"[OK]   IDs excluded from features, client_id used only for split grouping")


=== LEAKAGE AUDIT (final feature set used in Section 2) ===

1. Target-derived and future-window columns
   Forbidden set: ['clicks_last_30d', 'clicks_prev_30d', 'client_id', 'content_id', 'impressions_last_30d', 'impressions_prev_30d', 'sessions_last_30d', 'sessions_prev_30d', 'trend_direction', 'trend_pct']
   Present in features used: none
   Check passed: True

2. Train-without-suspect test on top feature: log_impressions_90d
   average_precision WITH log_impressions_90d: 0.607
   average_precision WITHOUT log_impressions_90d: 0.600
   Collapse toward base rate (0.517)? no large collapse -- not a leakage signature

3. Time-window leakage check
   content_age_days and days_since_last_update are computed once, as of the CSV's
   snapshot date -- they are legitimate features for THIS static dataset because
   they are always knowable before the label window (trend_direction looks at the
   trailing 30 vs prior 30 days, which is fully inside the 90-day snapshot window).
   Risk if this

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [4]:
print("=== CLAIM REWRITE ===\n")

precision_before = comparison.iloc[0]["precision_at_50"]
precision_after = comparison.iloc[1]["precision_at_50"]
base_rate_after = comparison.iloc[1]["test_base_rate"]

print("Original bold claim (how I would have stated the Week-5 result before this audit):")
print(
    f'  "Logistic Regression beats the baseline rule and reliably finds declining pages: '
    f'precision@50 of {precision_before:.2f} proves the model works."'
)
print()
print("Rewritten (safe language -- observed, measured, directional, decision-support):")
print(
    "  \"On a random row split, the model showed precision@50 of "
    f"{precision_before:.2f} against a test-set base rate of {base_rate_after:.2f}. "
    "Under a grouped split that holds out entire clients, the same model measured "
    f"precision@50 of {precision_after:.2f}. The gap between those two numbers is directional "
    "evidence that part of the random-split score came from the model recognizing clients "
    "it had already seen, not from patterns that transfer to a new client. The grouped-split "
    "number is the more honest estimate, and even that number is decision-support for "
    "ranking a review queue -- not proof the model will hold on every future client or "
    "time period.\""
)
print()

top_feature_display = top_feature_clean
print(f"Another bold claim (from feature coefficients, top feature: {top_feature_display}):")
print(
    f'  "{top_feature_display} drives the model -- it is the reason pages get flagged as declining."'
)
print()
print("Rewritten (safe language):")
print(
    f"  \"In the fitted logistic regression model (grouped split), {top_feature_display} had the "
    "largest-magnitude coefficient among the measured features. This is a descriptive finding "
    "about this fitted model on this dataset: it suggests a directional association between "
    f"{top_feature_display} and the model's score, not a causal claim about what drives decline, "
    "and not a guarantee the same feature will rank first on a different client mix or time period.\""
)


=== CLAIM REWRITE ===

Original bold claim (how I would have stated the Week-5 result before this audit):
  "Logistic Regression beats the baseline rule and reliably finds declining pages: precision@50 of 0.88 proves the model works."

Rewritten (safe language -- observed, measured, directional, decision-support):
  "On a random row split, the model showed precision@50 of 0.88 against a test-set base rate of 0.52. Under a grouped split that holds out entire clients, the same model measured precision@50 of 0.76. The gap between those two numbers is directional evidence that part of the random-split score came from the model recognizing clients it had already seen, not from patterns that transfer to a new client. The grouped-split number is the more honest estimate, and even that number is decision-support for ranking a review queue -- not proof the model will hold on every future client or time period."

Another bold claim (from feature coefficients, top feature: log_impressions_90d):
 

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.